# 模板与静态资源

学习目标：把学习记录渲染为列表页和详情页，生成可访问的链接与样式地址，并确认外部文本经过 HTML 自动转义。

前置知识：HTML 元素与属性、CSS 基本声明、URL、Python 字典与文件读写、FastAPI 路由。

适用版本：FastAPI 0.141.1、Starlette 1.6、Jinja2 3.1。TemplateResponse 使用 request、name 和 context 关键字参数。

环境准备：[FastAPI 环境与运行入口](README.md)。

工作目录：content/Web与应用开发/FastAPI。Notebook 先通过 TestClient 在应用内调用，临时模板和样式由本篇创建并在末尾删除；最后的浏览器实验使用端口 8230，只读取两条固定记录。

配套脚本：位于 scripts/23-templates-and-static-files/，仅用于真实浏览器实验。

（1）[app.py](scripts/23-templates-and-static-files/app.py)：组合本篇已讲解的页面路由、固定数据与静态资源。

（2）[list.html](scripts/23-templates-and-static-files/templates/list.html) 和 [detail.html](scripts/23-templates-and-static-files/templates/detail.html)：完整列表页与详情页模板。

（3）[style.css](scripts/23-templates-and-static-files/static/style.css)：限制行宽、设置行距并允许长文本换行。

## 1 把一个值填入 HTML

模板是带有变量和控制语句的文本。Jinja 渲染模板时，把上下文（context）中的值填入相应位置，生成最终字符串。下面的 title 是传入的标题；双花括号表示输出表达式。

Environment 保存模板配置。先从字符串创建一个最小模板，显式开启 HTML 自动转义，再调用 render；此时还没有 HTTP 请求。

In [1]:
from jinja2 import Environment

environment = Environment(autoescape=True)
greeting = environment.from_string("<h1>{{ title }}</h1>")
html = greeting.render(title="今天的学习记录")
assert html == "<h1>今天的学习记录</h1>"
print(html)  # 预期：<h1>今天的学习记录</h1>，此时只是字符串。

<h1>今天的学习记录</h1>


## 2 用 Jinja2Templates 返回页面

Jinja2Templates 连接模板引擎与 Web 请求。directory 指向模板目录；TemplateResponse 接收当前 Request、模板文件名和上下文字典，再返回 HTML 响应。路由声明 response_class=HTMLResponse，也会让 OpenAPI 标明成功响应的 HTML 类型。

先为这次 Notebook 运行创建独立临时目录，把刚才的片段保存成 hello.html。后续新模板也放在这里，避免依赖已有文件。

In [2]:
from pathlib import Path
from tempfile import TemporaryDirectory

from fastapi.templating import Jinja2Templates

workspace = TemporaryDirectory(prefix="fastapi-templates-")
work_path = Path(workspace.name)
template_path = work_path / "templates"
template_path.mkdir()
(template_path / "hello.html").write_text(
    "<h1>{{ title }}</h1>", encoding="utf-8"
)
templates = Jinja2Templates(directory=template_path)

下面只返回一段 HTML，便于观察模板与响应的连接方式。TestClient 额外提供 template 和 context 属性，可以检查选中了哪个模板、传入了哪些数据；这些属性是应用内测试信息，不是发给浏览器的 HTTP 字段。

In [3]:
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import HTMLResponse
from fastapi.testclient import TestClient

app = FastAPI()


@app.get("/hello", response_class=HTMLResponse)
def hello(request: Request):
    return templates.TemplateResponse(
        request=request, name="hello.html", context={"title": "今天的学习记录"}
    )


with TestClient(app) as client:
    response = client.get("/hello")
    assert response.template.name == "hello.html"
    assert response.context["title"] == "今天的学习记录"
    print(response.status_code, response.headers["content-type"])  # 预期：200 text/html; charset=utf-8。
    print(response.text)  # 预期：<h1>今天的学习记录</h1>。

200 text/html; charset=utf-8
<h1>今天的学习记录</h1>


C:\Users\ZHUANG\miniconda3\envs\hands-on-computing\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 3 挂载一份静态样式

StaticFiles 发送指定目录里的公开文件，不执行 Jinja 模板。浏览器取得页面 HTML 后，会另发请求加载其中引用的 CSS，因此页面返回成功不代表样式地址也正确。

![HTML 页面与 CSS 是两次请求](image/illustration/23-01-template-assets.svg)

图示：页面与静态样式的两次请求；CSS 地址从返回的 HTML 中读取，资源挂载只公开指定目录。

下面先创建目录，再将其挂载为 /static 并命名 static，随后可用 url\_for 生成资源地址。CSS 只设置本例行宽、行距与长文本换行；模板与公开样式分开存放。先单独验证 style.css，再到浏览器检查页面与样式。

In [4]:
from fastapi.staticfiles import StaticFiles

static_path = work_path / "static"
static_path.mkdir()
style = """body {
  max-width: 44rem;
  margin: 2rem auto;
  padding: 0 1rem;
  font-family: system-ui, sans-serif;
  line-height: 1.6;
}

.record-note {
  overflow-wrap: anywhere;
}
"""
(static_path / "style.css").write_text(style, encoding="utf-8")
app.mount("/static", StaticFiles(directory=static_path), name="static")

读取 CSS 和读取页面是两次请求。先单独确认样式地址可用，再在 HTML 中引用；没有对应文件时，静态资源请求返回 404。

In [5]:
with TestClient(app) as client:
    css = client.get("/static/style.css")
    missing_css = client.get("/static/missing.css")
    assert css.status_code == 200
    assert css.headers["content-type"].startswith("text/css")
    assert css.text.splitlines() == style.splitlines()
    assert missing_css.status_code == 404
    print(css.status_code, css.headers["content-type"])  # 预期：200 text/css; charset=utf-8。
    print("不存在的资源：", missing_css.status_code)  # 预期：不存在的资源： 404。

200 text/css; charset=utf-8
不存在的资源： 404


## 4 列表模板与 URL 生成

先给出两条独立的固定记录。第二条内容刻意带有看似可执行的 HTML，用于后面的自动转义观察。数据在路由中准备，模板只负责显示；不要在模板中执行数据库查询或网络调用。

In [6]:
records = [
    {"id": 1, "title": "模板上下文", "note": "用字典把学习内容传入页面。"},
    {
        "id": 2,
        "title": "自动转义实验",
        "note": "<script>alert('demo')</script><b>学习记录</b>",
    },
]

for 语句逐项取出 records，每项临时命名为 record；record.title 和 record.id 分别读取记录的标题和编号。与输出表达式不同，控制语句使用花括号加百分号。

模板中的 url_for 按路由名称生成 URL。record_detail 是详情路由名，record_id 提供该路由的路径参数；static 是挂载名称，path 指向其中的文件。链接地址来自应用路由，不需要在模板里写死主机和端口。

这次使用完整 HTML 文档。head 内的 link 元素声明样式表，浏览器加载它的 href 地址。

In [7]:
# 列表模板使用当前请求生成静态资源和详情页 URL，循环逐项渲染 records。
list_html = """<!doctype html>
<html lang="zh-CN">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>{{ title }}</title>
  <link rel="stylesheet" href="{{ url_for('static', path='/style.css') }}">
</head>
<body>
  <main>
    <h1>{{ title }}</h1>
    <p>选择一条记录查看详情。</p>
    <ul>
      {% for record in records %}
      <li><a href="{{ url_for('record_detail', record_id=record.id) }}">{{ record.title }}</a></li>
      {% endfor %}
    </ul>
  </main>
</body>
</html>"""
# 预期：513，等于当前列表模板的字符数；模板写入 list.html。
(template_path / "list.html").write_text(list_html, encoding="utf-8")

513

定义列表路由，把页面标题和记录列表作为上下文传入。这里显式指定路由名 record_list，方便详情页生成“返回列表”的地址。列表模板引用的详情路由会在下一节注册，然后一起调用。

In [8]:
@app.get("/", response_class=HTMLResponse, name="record_list")
def record_list(request: Request):
    return templates.TemplateResponse(
        request=request,
        name="list.html",
        context={"title": "学习记录", "records": records},
    )

## 5 详情页与不存在的记录

详情模板只接收一个 record，并通过 url_for('record_list') 生成返回链接。模板决定如何显示；记录是否存在，由路由先判断。

In [9]:
detail_html = """<!doctype html>
<html lang="zh-CN">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>{{ record.title }} · 学习记录</title>
  <link rel="stylesheet" href="{{ url_for('static', path='/style.css') }}">
</head>
<body>
  <main>
    <p><a href="{{ url_for('record_list') }}">返回列表</a></p>
    <h1>{{ record.title }}</h1>
    <p>记录编号：{{ record.id }}</p>
    <p class="record-note">{{ record.note }}</p>
  </main>
</body>
</html>"""
# 预期：479，等于当前详情模板的字符数；模板写入 detail.html。
(template_path / "detail.html").write_text(detail_html, encoding="utf-8")

479

找不到记录时抛出 HTTPException(404)。本例保留 FastAPI 的默认错误响应，所以成功详情是 HTML，404 错误体是 JSON；需要 HTML 错误页时，应另行设计对应的错误处理器和模板。

In [10]:
@app.get("/records/{record_id}", response_class=HTMLResponse, name="record_detail")
def record_detail(request: Request, record_id: int):
    # 找到记录就渲染详情；遍历结束仍未找到时，交给 HTTP 404 处理。
    for record in records:
        if record["id"] == record_id:
            return templates.TemplateResponse(
                request=request, name="detail.html", context={"record": record}
            )
    raise HTTPException(status_code=404, detail="记录不存在")


# 从页面链接、模板上下文和不存在的记录三方面观察实际响应。
with TestClient(app) as client:
    page = client.get("/")
    detail = client.get("/records/1")
    missing = client.get("/records/999")
    assert 'href="http://testserver/records/1"' in page.text
    assert 'href="http://testserver/static/style.css"' in page.text
    assert 'href="http://testserver/"' in detail.text
    assert detail.context["record"]["id"] == 1
    assert missing.status_code == 404
    print(page.template.name, len(page.context["records"]))  # 预期：list.html 2。
    print(detail.template.name, detail.context["record"]["title"])  # 预期：detail.html 模板上下文。
    print(missing.status_code, missing.json())  # 预期：404 {'detail': '记录不存在'}。

list.html 2
detail.html 模板上下文
404 {'detail': '记录不存在'}


## 6 让外部文本保持为文本

用 directory 创建 Jinja2Templates 时，Starlette 默认对 .html、.htm 和 .xml 模板启用自动转义。变量里的小于号等字符会变为 HTML 实体，使浏览器把它们当成文字显示。这里观察的是 HTML 文本位置，不把外部输入放入脚本或事件属性。

转义发生在生成 HTML 时，原始记录不会因此被改写。本例不使用 safe 过滤器或 Markup 把外部内容标为可信 HTML。模板源码也始终由应用提供，只把记录作为上下文值传入。

In [11]:
with TestClient(app) as client:
    escaped = client.get("/records/2")
    assert escaped.status_code == 200
    assert "<script>" not in escaped.text
    assert "<b>学习记录</b>" not in escaped.text
    assert "&lt;script&gt;" in escaped.text
    assert "&lt;b&gt;学习记录&lt;/b&gt;" in escaped.text
    assert escaped.context["record"]["note"] == records[1]["note"]
    note_line = next(line for line in escaped.text.splitlines() if "record-note" in line)
    print(note_line.strip())  # 预期：record-note 段落内的 <script>、<b> 写成 &lt;…&gt; 实体，浏览器显示原始文字。

<p class="record-note">&lt;script&gt;alert(&#39;demo&#39;)&lt;/script&gt;&lt;b&gt;学习记录&lt;/b&gt;</p>


## 7 同一份数据，HTML 与 JSON 各自返回

服务端渲染把数据放入模板，生成供浏览器阅读的页面。JSON API 返回结构化数据，便于程序继续处理；模板本身不会自动生成对应的 JSON 接口，需要声明独立路由。

下面两个入口使用同一份固定记录。程序读取 JSON 后如果还要把值插入网页，仍须按目标位置安全显示，不能因为数据来自 JSON 就把它直接当成可信 HTML。

In [12]:
@app.get("/api/records")
def record_data() -> list[dict[str, int | str]]:
    return records


with TestClient(app) as client:
    html_response = client.get("/")
    json_response = client.get("/api/records")
    assert html_response.status_code == json_response.status_code == 200
    assert json_response.json() == records
    assert "<script>" in json_response.json()[1]["note"]
    print("页面：", html_response.headers["content-type"])  # 预期：页面： text/html; charset=utf-8。
    print("数据：", json_response.headers["content-type"])  # 预期：数据： application/json。
    print("JSON 记录数：", len(json_response.json()))  # 预期：JSON 记录数： 2。

页面： text/html; charset=utf-8
数据： application/json
JSON 记录数： 2


## 8 浏览器打开列表、详情和样式

配套应用只组合本篇的两条记录、列表路由、详情路由、JSON 路由和样式挂载。app.py 用自身所在目录定位 templates 和 static，因此资源定位不依赖终端恰好位于脚本旁边。它不读写数据库，也不需要运行前面的 Notebook 才能启动。

Step 1：在课程环境中，从项目根目录进入本课程目录。

```powershell
Set-Location 'content/Web与应用开发/FastAPI'
```

Step 2：在独立终端启动本地服务，并等待出现 Application startup complete。

```powershell
python -m uvicorn app:app --app-dir scripts/23-templates-and-static-files --host 127.0.0.1 --port 8230
```

浏览器打开 http://127.0.0.1:8230/，确认列表有“模板上下文”和“自动转义实验”两条链接。点击第一条，查看编号 1 与内容，再点击“返回列表”。点击第二条，确认尖括号、script 和 b 都作为文字出现，没有弹窗，学习记录也没有因为样本文本而变粗。

![自动转义后的详情页：script 和 b 标签作为文字显示](image/23-template-escaping.png)

图：详情页保留样本文本中的尖括号，内容没有变成可执行脚本或加粗元素。

在浏览器开发者工具的 Network 面板刷新页面，确认 /static/style.css 成功加载且类型为 CSS；在 Elements 面板查看详情的 .record-note，里面应是一个文本内容，不含 script 或 b 子元素。缩窄窗口后，文字仍应完整显示。

浏览器打开 http://127.0.0.1:8230/api/records，检查两条 JSON 记录；打开 http://127.0.0.1:8230/records/999，确认响应为 404 和“记录不存在”。页面路由返回 HTML，不代表错误体也自动变为 HTML。

Step 3：在运行服务的终端按 Ctrl+C，等待应用关闭并回到命令提示符。

Step 4：关闭本次浏览器标签页。

配套文件保留供重复实验使用；这个服务没有需要删除的数据文件。Notebook 的临时资源在下一单元清理，如果中途停止执行，也可运行该单元完成清理。

In [13]:
# 所有 TestClient 都已退出 with，此时不再请求使用这些文件的应用。
workspace.cleanup()
assert not work_path.exists()
print("Notebook 临时模板和样式已删除。")  # 预期：Notebook 临时模板和样式已删除。

Notebook 临时模板和样式已删除。


## 本章小结

（1）Jinja 使用上下文值渲染模板；TemplateResponse 把当前请求、模板和数据组合成 HTML 响应。

（2）url_for 使用路由名和参数生成地址，StaticFiles 则发送独立目录中的公开文件。

（3）HTML 自动转义让外部文本按文字显示；不要把外部值标为可信 HTML，也不要让外部输入成为模板源码。

（4）HTML 页面、JSON 接口与错误响应分别设计；数据准备放在路由，模板承担显示工作。

自查：模板里的 record_id 从哪里来？HTML 源码出现实体时，为什么浏览器仍显示尖括号？CSS 地址返回 404 时，应检查挂载名称、资源路径还是数据记录？

## 练习

Notebook 练习先重新运行到对应示例，在末尾清理前完成；使用配套服务修改文件后，重启服务再检查。完成后关闭服务并清理临时目录。

（1）增加一条编号 3 的固定记录。验证列表出现第三条链接，点击后显示编号 3，且 /api/records 也包含同一条数据。

（2）在列表模板中增加“暂无学习记录”的空列表分支。分别传入两条记录和空列表，验证只有空列表时显示提示，且没有详情链接。

（3）把详情路径改为 /notes/{record_id}，保留路由名称 record_detail。验证列表链接自动指向新路径，点击可用，模板内无需写入新的固定路径。

（4）把一条 note 改成含 &lt;img src=x onerror=alert('demo')&gt; 的文字。验证响应中尖括号已转义，浏览器显示完整样本文本，详情容器中没有 img 元素。

提示：空列表可用 Jinja 的 if/else 判断；URL 生成依赖路由名称和参数；转义练习只改变上下文值，保留自动转义配置。

## 参考与引用来源

- **FastAPI 官方文档**：[Templates](https://fastapi.tiangolo.com/advanced/templates/)，定位 Using Jinja2Templates、Template Context Values、Template url_for Arguments、Templates and static files；[Custom Response](https://fastapi.tiangolo.com/advanced/custom-response/)，定位 JSON Responses 与 HTML Response；[Handling Errors](https://fastapi.tiangolo.com/tutorial/handling-errors/#use-httpexception)，定位 HTTPException 的状态码和默认 JSON 错误体。
- **Starlette 官方文档**：[Templates](https://starlette.dev/templates/)，定位 Jinja2Templates、Autoescape、Testing template responses、Asynchronous template rendering；[Static Files](https://starlette.dev/staticfiles/)，定位 directory、check_dir、挂载与不存在文件的响应。Jinja2Templates 的目录加载和自动转义按本篇 Starlette 1.6 核查。
- **Jinja 官方文档**：[API](https://jinja.palletsprojects.com/en/stable/api/)，定位 Environment、from_string、render、FileSystemLoader 与 Autoescaping；[Template Designer Documentation](https://jinja.palletsprojects.com/en/stable/templates/)，定位 Variables、For、If、Working with Automatic Escaping；[Sandbox 的 Security Considerations](https://jinja.palletsprojects.com/en/stable/sandbox/#security-considerations)，用于区分传入普通文本与执行不可信模板，本篇只使用应用提供的模板。
- **Python 官方文档**：[tempfile.TemporaryDirectory](https://docs.python.org/3.12/library/tempfile.html#tempfile.TemporaryDirectory) 的 cleanup；[pathlib](https://docs.python.org/3.12/library/pathlib.html)，定位 Path.write_text、Path.mkdir、Path.resolve 与路径连接。
- **WHATWG HTML 标准**：[link 元素](https://html.spec.whatwg.org/multipage/semantics.html#the-link-element) 与 [stylesheet 链接类型](https://html.spec.whatwg.org/multipage/links.html#link-type-stylesheet)，用于样式表 URL 与浏览器加载行为。
- **W3C／CSSWG**：[CSS 2.2 视觉格式化细节](https://www.w3.org/TR/CSS22/visudet.html)，定位 max-width 和 line-height；[CSS Text Level 3 §5.4](https://www.w3.org/TR/css-text-3/#overflow-wrap-property)，用于配套样式的长文本换行，该模块为候选推荐草案，页面行为另经浏览器检查。
- **OWASP 官方备忘单**：[Cross Site Scripting Prevention](https://cheatsheetseries.owasp.org/cheatsheets/Cross_Site_Scripting_Prevention_Cheat_Sheet.html)，定位 Output Encoding for HTML Contexts、Dangerous Contexts 与 Safe Sinks，说明按输出位置处理数据的必要性。